In [ ]:
# ============================================================
# Fine_tune_DARTIS_OSD.ipynb

# Hard-negative fine-tuning script for Sentinel-1 SAR oil spill segmentation.

# This script fine-tunes a DeepLabV3-ResNet50 model trained on Refined SOS
# using additional hard-negative samples from DARTIS and CSIRO OSD.

# Classes:
#     0 = Background / Look-alike / No-oil
#     1 = Oil spill

# The DARTIS dataset does not follow a specific directory hierarchy.
# Image files are identified by filename prefixes:
#
#   NW = No Wind
#   NC = Natural Look-Alike
#   OW = Oil on Water
#   OC = Oil Close to Coast
#
# Only NW and NC samples are used in this study as hard-negative examples.
# All selected samples are assigned background masks and treated as non-oil
# examples during fine-tuning.

# OSD_Dataset structure:
# data_root/
# ├── S1SAR_UnBalanced_400by400_Class_0/
# │   ├── 0/
# └── S1SAR_UnBalanced_400by400_Class_1/
#     ├── 1/
# ============================================================
import os
import shutil
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset, WeightedRandomSampler
from torchvision.models.segmentation import deeplabv3_resnet50

from google.colab import drive
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# 1. Copy datasets to local /content
# ============================================================

drive_base = "/content/drive/MyDrive/SAR_AI_Oil_Spill_Detection"
local_base = "/content/SAR_AI_Oil_Spill_Detection"

os.makedirs(local_base, exist_ok=True)

for folder in ["refined SOS", "DARTIS", "OSD"]:
    src = os.path.join(drive_base, folder)
    dst = os.path.join(local_base, folder)

    if os.path.exists(dst):
        print(f"Already exists, skip: {folder}")
    else:
        print(f"Copying {folder}...")
        shutil.copytree(src, dst)
        print(f"Copied: {folder}")

os.makedirs(local_base, exist_ok=True)

print("Copied datasets to local /content")

# ============================================================
# 2. Paths
# ============================================================

rootDir = os.path.join(local_base, "refined SOS")
dartisDir = os.path.join(local_base, "DARTIS")

# OSD no-oil path
osdDir = os.path.join(
    local_base,
    "OSD",
    "S1SAR_UnBalanced_400by400_Class_0",
    "0"
)

baseModelPath = os.path.join(
    drive_base,
    "DeeplabV3_baseline.pth"
)

savePath = os.path.join(
    drive_base,
    "DeeplabV3_finetuned.pth"
)

trainImageDir = os.path.join(rootDir, "images", "train")
trainMaskDir  = os.path.join(rootDir, "masks",  "train")

# ============================================================
# 3. Refined SOS Dataset
# ============================================================

class RefinedSOSDataset(Dataset):

    def __init__(self, image_dir, mask_dir):

        self.image_dir = image_dir
        self.mask_dir = mask_dir

        self.image_files = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff"))
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):

        img_name = self.image_files[idx]

        img_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        image = Image.open(img_path).convert("L")
        image = image.resize((256, 256))
        image = np.array(image, dtype=np.float32) / 255.0
        image = np.stack([image, image, image], axis=0)
        image = torch.from_numpy(image).float()

        mask = Image.open(mask_path).convert("L")
        mask = mask.resize((256, 256), resample=Image.NEAREST)
        mask = np.array(mask)
        mask = (mask > 0).astype(np.uint8)
        mask = torch.from_numpy(mask).long()

        return image, mask

# ============================================================
# 4. DARTIS Negative Dataset
# Only NW (No Wind) and NC (Natural Look-Alike) samples were used.
# ============================================================

class DARTISNegativeDataset(Dataset):

    def __init__(self, image_dir):

        self.image_dir = image_dir

        self.image_files = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff"))
            and os.path.basename(f).lower().startswith(("nw", "nc"))
        ])

        print("DARTIS nw/nc negative samples:", len(self.image_files))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):

        img_name = self.image_files[idx]
        img_path = os.path.join(self.image_dir, img_name)

        image = Image.open(img_path).convert("L")
        image = image.resize((256, 256))
        image = np.array(image, dtype=np.float32) / 255.0
        image = np.stack([image, image, image], axis=0)
        image = torch.from_numpy(image).float()

        mask = torch.zeros((256, 256), dtype=torch.long)

        return image, mask

# ============================================================
# 5. OSD Negative Dataset
# Class 0 = no-oil / clean sea / lookalike
# All samples were assigned background masks and treated as non-oil examples during training.
# ============================================================

class OSDNegativeDataset(Dataset):

    def __init__(self, image_dir):

        self.image_dir = image_dir

        self.image_files = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff"))
        ])

        print("OSD no-oil/lookalike negative samples:", len(self.image_files))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):

        img_name = self.image_files[idx]
        img_path = os.path.join(self.image_dir, img_name)

        image = Image.open(img_path).convert("L")
        image = image.resize((256, 256))
        image = np.array(image, dtype=np.float32) / 255.0
        image = np.stack([image, image, image], axis=0)
        image = torch.from_numpy(image).float()

        # OSD Class 0 contains no-oil/look-alike samples and is treated as background.
        mask = torch.zeros((256, 256), dtype=torch.long)

        return image, mask

# ============================================================
# 6. Dataset mixing
# ============================================================

sos_dataset = RefinedSOSDataset(trainImageDir, trainMaskDir)
dartis_dataset = DARTISNegativeDataset(dartisDir)
osd_dataset = OSDNegativeDataset(osdDir)

mixed_dataset = ConcatDataset([
    sos_dataset,
    dartis_dataset,
    osd_dataset
])

print("Refined SOS samples:", len(sos_dataset))
print("DARTIS negative samples:", len(dartis_dataset))
print("OSD negative samples:",len(osd_dataset))
print("Total mixed samples:", len(mixed_dataset))

sos_weight = 5
dartis_weight = 1.0
osd_weight = 1.0

weights = (
    [sos_weight] * len(sos_dataset) +
    [dartis_weight] * len(dartis_dataset) +
    [osd_weight] * len(osd_dataset)
)

sampler = WeightedRandomSampler(
    weights=weights,
    num_samples=len(sos_dataset),
    replacement=True
)

loader = DataLoader(
    mixed_dataset,
    batch_size=16,
    sampler=sampler,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

# ============================================================
# 7. Load base DeepLabV3 model
# ============================================================

model = deeplabv3_resnet50(
    weights=None,
    weights_backbone=None,
    num_classes=2,
    aux_loss=False
)

model.load_state_dict(
    torch.load(baseModelPath, map_location=device)
)

model = model.to(device)

print("Base Refined SOS model loaded.")

# ============================================================
# 8. Loss functions
# ============================================================

class DiceLoss(nn.Module):

    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):

        probs = torch.softmax(logits, dim=1)[:, 1, :, :]
        targets = targets.float()

        intersection = (probs * targets).sum()
        union = probs.sum() + targets.sum()

        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)

        return 1.0 - dice

ce_loss = nn.CrossEntropyLoss(
    weight=torch.tensor([1.0, 1.5], device=device)
)

dice_loss = DiceLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-5
)

# ============================================================
# 9. Mixed fine-tuning
# ============================================================

num_epochs = 5

for epoch in range(num_epochs):

    model.train()
    running_loss = 0.0

    loop = tqdm(loader)

    for images, masks in loop:

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)["out"]

        loss_ce = ce_loss(outputs, masks)
        loss_dice = dice_loss(outputs, masks)

        loss = loss_ce + loss_dice

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        loop.set_description(f"Epoch [{epoch+1}/{num_epochs}]")
        loop.set_postfix(
            loss=loss.item(),
            ce=loss_ce.item(),
            dice=loss_dice.item()
        )

    epoch_loss = running_loss / len(loader)
    print(f"Epoch {epoch+1} Loss: {epoch_loss:.4f}")

# ============================================================
# 10. Save model
# ============================================================

torch.save(model.state_dict(), savePath)

print("Saved:", savePath)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Already exists, skip: refined SOS
Already exists, skip: DARTIS
Already exists, skip: OSD
Copied datasets to local /content
DARTIS nw/nc negative samples: 2290
OSD no-oil/lookalike negative samples: 3725
Refined SOS samples: 6455
DARTIS negative samples: 2290
OSD negative samples: 3725
Total mixed samples: 12470
Base Refined SOS model loaded.


Epoch [1/5]: 100%|██████████| 404/404 [09:30<00:00,  1.41s/it, ce=0.132, dice=0.125, loss=0.257]


Epoch 1 Loss: 0.3693


Epoch [2/5]: 100%|██████████| 404/404 [09:39<00:00,  1.43s/it, ce=0.127, dice=0.105, loss=0.231]


Epoch 2 Loss: 0.3289


...

Epoch 3 Loss: 0.3006


Epoch [4/5]: 100%|██████████| 404/404 [09:39<00:00,  1.43s/it, ce=0.127, dice=0.112, loss=0.24]


Epoch 4 Loss: 0.2888


Epoch [5/5]: 100%|██████████| 404/404 [09:39<00:00,  1.43s/it, ce=0.173, dice=0.141, loss=0.314]


Epoch 5 Loss: 0.2747
Saved: /content/drive/MyDrive/SAR_AI_Oil_Spill_Detection/DeeplabV3_finetuned.pth
